Import the coherence, CoLA and LLM-rater and compare them on imported prompt-based text and paraphrases.

# Loading text data

In [5]:
from pathlib import Path
import json
import torch

def load_json_lists_under(folder: Path) -> dict:
    """
    Recursively load all *.json files under `folder`.
    Returns a dict: { "relative/path/stem": [list_of_strings], ... }
    """
    data = {}
    for p in folder.rglob("*.json"):
        try:
            with open(p, "r", encoding="utf-8") as f:
                contents = json.load(f)
            if isinstance(contents, list) and all(isinstance(x, str) for x in contents):
                key = str(p.relative_to(folder).with_suffix(""))  # relative path without .json
                data[key] = contents
        except Exception as e:
            print(f"Skipping {p}: {e}")
    return data


root = Path("../model-comparison/data/")

for folder in root.iterdir():
    if folder.is_dir():
        var_name = f"{folder.name}"
        globals()[var_name] = load_json_lists_under(folder)
        print(f"Loaded {len(globals()[var_name])} JSON files into `{var_name}`")


Loaded 9 JSON files into `shuffled_words`
Loaded 9 JSON files into `shuffled_sentences`
Loaded 9 JSON files into `shuffled_tokens`
Loaded 9 JSON files into `clean`


In [30]:
key_map = {

    "viktor": {
        "clean": "viktor_100_narratives",
        "tokens": "viktor_shuffled_tokens",
        "words": "viktor_shuffled_words",
        "sentences": "viktor_shuffled_sentences"
    },

    "prague": {
        "clean": "prague_100_narratives",
        "tokens": "prague_shuffled_tokens",
        "words": "prague_shuffled_words",
        "sentences": "prague_shuffled_sentences"
    },

    "sciencefic": {
        "clean": "sciencefic_100_narratives",
        "tokens": "sciencefic_shuffled_tokens",
        "words": "sciencefic_shuffled_words",
        "sentences": "sciencefic_shuffled_sentences"
    },

    "gpt4_para1": {
        "clean": "gpt4_para1",
        "tokens": "gpt4_para1_tokens_shuffled",
        "words": "gpt4_para1_words_shuffled",
        "sentences": "gpt4_para1_sentences_shuffled"
    },

    "gpt4_para2": {
        "clean": "gpt4_para2",
        "tokens": "gpt4_para2_tokens_shuffled",
        "words": "gpt4_para2_words_shuffled",
        "sentences": "gpt4_para2_sentences_shuffled"
    },

    "gpt4_para3": {
        "clean": "gpt4_para3",
        "tokens": "gpt4_para3_tokens_shuffled",
        "words": "gpt4_para3_words_shuffled",
        "sentences": "gpt4_para3_sentences_shuffled"
    },

    "gpt5_para1": {
        "clean": "gpt5_para1",
        "tokens": "gpt5_para1_tokens_shuffled",
        "words": "gpt5_para1_words_shuffled",
        "sentences": "gpt5_para1_sentences_shuffled"
    },

    "gpt5_para2": {
        "clean": "gpt5_para2",
        "tokens": "gpt5_para2_tokens_shuffled",
        "words": "gpt5_para2_words_shuffled",
        "sentences": "gpt5_para2_sentences_shuffled"
    },

    "gpt5_para3": {
        "clean": "gpt5_para3",
        "tokens": "gpt5_para3_tokens_shuffled",
        "words": "gpt5_para3_words_shuffled",
        "sentences": "gpt5_para3_sentences_shuffled"
    }
}

# Obtaining values from other models

## The Coherence model - sgnlp_coherence 

In [2]:
from sgnlp.models.coherence_momentum import CoherenceMomentumModel, CoherenceMomentumConfig, \
    CoherenceMomentumPreprocessor

# Load Model
config = CoherenceMomentumConfig.from_pretrained(
    "https://storage.googleapis.com/sgnlp-models/models/coherence_momentum/config.json"
)
model = CoherenceMomentumModel.from_pretrained(
    "https://storage.googleapis.com/sgnlp-models/models/coherence_momentum/pytorch_model.bin",
    config=config
)

preprocessor = CoherenceMomentumPreprocessor(config.model_size, config.max_len)

/home/jdias/miniconda3/envs/sgnlp_coherence/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jdias/miniconda3/envs/sgnlp_coherence/lib/python3.11/site-packages/transformers/utils/hub.py:580: FutureWarning: Using `from_pretrained` with the url of a file (here https://storage.googleapis.com/sgnlp-models/models/coherence_momentum/config.json) is deprecated and won't be possible anymore in v5 of Transformers. You should host your file on the Hub (hf.co) instead and use the repository ID. Note that this is not compatible with the caching system (your file will be downloaded at each execution) or multiple processes (each process will download the file in a different temporary file).
  warnings.warn(
/home/jdias/miniconda3/envs/sgnlp_coherence/lib/python3.11/site-packages/transformers/utils/hub.py:580: Fu

In [ ]:
import torch
model.eval()

def score_texts_batched(texts, batch_size=32):
    """Return list[float] of scores for a list of strings using batched inference."""
    scores = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start+batch_size]
            tensors = preprocessor(batch)
            # tensors["tokenized_texts"] should be [B, ...]
            batch_scores = model.get_main_score(tensors["tokenized_texts"])
            # Convert to Python floats
            scores.extend(batch_scores.detach().cpu().tolist())
    return scores

BATCH_SIZE = 128

results = {}

for clean_key, related_keys in key_map.items():
    # Fetch aligned lists (each len == 100 per your setup)
    clean_texts     = clean[clean_key]
    token_texts     = shuffled_tokens[related_keys["tokens"]]
    word_texts      = shuffled_words[related_keys["words"]]
    sentence_texts  = shuffled_sentences[related_keys["sentences"]]

    n = len(clean_texts)
    assert len(token_texts) == n and len(word_texts) == n and len(sentence_texts) == n, \
        f"Length mismatch for key {clean_key}"

    # Concatenate all four streams so we only call preprocessor/model once per loop
    all_texts = clean_texts + token_texts + word_texts + sentence_texts #order is kept

    # Batched scoring (vectorized across the 4*n texts)
    all_scores = score_texts_batched(all_texts, batch_size=BATCH_SIZE)

    # Split them back into their groups
    clean_scores     = all_scores[0*n : 1*n]
    token_scores     = all_scores[1*n : 2*n]
    word_scores      = all_scores[2*n : 3*n]
    sentence_scores  = all_scores[3*n : 4*n]

    # Store
    results[clean_key] = {
        "clean": clean_scores,
        "tokens": token_scores,
        "words": word_scores,
        "sentences": sentence_scores
    }

    #progress
    print(f"{clean_key}: done. Examples:",
          clean_scores[0], token_scores[0], word_scores[0], sentence_scores[0])


viktor: done. Examples: 11.936232566833496 -26.200389862060547 -26.422626495361328 -1.6471409797668457
prague: done. Examples: 19.50853729248047 -28.530275344848633 -28.441946029663086 -16.485057830810547
sciencefic: done. Examples: 10.367465019226074 -29.009567260742188 -28.1842041015625 -9.786478996276855
gpt4_para1: done. Examples: 17.99173927307129 -26.217805862426758 -26.39042854309082 -18.92676544189453
gpt4_para2: done. Examples: 16.032718658447266 -20.609649658203125 -17.36447525024414 -13.433463096618652
gpt4_para3: done. Examples: 18.807987213134766 -26.24340057373047 -25.26621437072754 -9.653212547302246
gpt5_para1: done. Examples: 18.455106735229492 -27.573211669921875 -27.65407371520996 -10.202364921569824
gpt5_para2: done. Examples: 14.837810516357422 -16.24241065979004 -9.653375625610352 -12.051663398742676
gpt5_para3: done. Examples: 12.992559432983398 -25.762344360351562 -25.844669342041016 -17.742631912231445


Save for later

In [45]:
import pickle

with open("../model-comparison/results/the_coherence.pkl", "wb") as f:
    pickle.dump(results, f)

## CoLA model - change kernel

In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("cointegrated/roberta-large-cola-krishna2020")
model = AutoModelForSequenceClassification.from_pretrained("cointegrated/roberta-large-cola-krishna2020")

def score(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
    return probs.tolist()[0]

print("Grammatical:", score("This is a perfectly normal sentence."))
print("Ungrammatical:", score("This a sentence not good is."))


Grammatical: [0.9904881715774536, 0.009511778131127357]
Ungrammatical: [0.010091143660247326, 0.9899088144302368]


So the Acceptable index is 0 and in index 1 is the prob of being ungrammatical.

In [31]:
# --- config ---
MODEL_NAME = "cointegrated/roberta-large-cola-krishna2020"
BATCH_SIZE = 32
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ACCEPT_IDX = 0  

# --- load model ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

@torch.no_grad()
def score_batch(texts):
    """Return list[float] of 'acceptable' probabilities for a list of strings."""
    out = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start+BATCH_SIZE]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        out.extend(probs[:, ACCEPT_IDX].detach().cpu().tolist())
    return out


# --- run over dicts ---
hf_results = {}

for clean_key, related in key_map.items():
    clean_texts    = clean[clean_key]
    token_texts    = shuffled_tokens[related["tokens"]]
    word_texts     = shuffled_words[related["words"]]
    sentence_texts = shuffled_sentences[related["sentences"]]

    n = len(clean_texts)
    all_texts = clean_texts + token_texts + word_texts + sentence_texts
    scores = score_batch(all_texts)

    hf_results[clean_key] = {
        "clean":     scores[0*n:1*n],
        "tokens":    scores[1*n:2*n],
        "words":     scores[2*n:3*n],
        "sentences": scores[3*n:4*n],
    }

    print(f"{clean_key}: done. First scores ->",
          hf_results[clean_key]['clean'][0],
          hf_results[clean_key]['tokens'][0],
          hf_results[clean_key]['words'][0],
          hf_results[clean_key]['sentences'][0])


viktor: done. First scores -> 0.990220308303833 0.020245017483830452 0.019679434597492218 0.9564821720123291
prague: done. First scores -> 0.9708409905433655 0.02416745387017727 0.02935520000755787 0.9441577196121216
sciencefic: done. First scores -> 0.9546310305595398 0.02358989231288433 0.01762944646179676 0.9509793519973755
gpt4_para1: done. First scores -> 0.9894797205924988 0.01756417378783226 0.018481776118278503 0.9808109402656555
gpt4_para2: done. First scores -> 0.9319757223129272 0.023325953632593155 0.019053732976317406 0.9011991024017334
gpt4_para3: done. First scores -> 0.9867134094238281 0.018329765647649765 0.027319613844156265 0.8848375082015991
gpt5_para1: done. First scores -> 0.9953719973564148 0.016621600836515427 0.016936715692281723 0.986598014831543
gpt5_para2: done. First scores -> 0.9855703115463257 0.018841387704014778 0.016112834215164185 0.8916875123977661
gpt5_para3: done. First scores -> 0.9899945855140686 0.029910365119576454 0.04170824959874153 0.8881634

In [33]:
import pickle 

with open("../model-comparison/results/cola.pkl", "wb") as f:
    pickle.dump(hf_results, f)